# Experiment 3 — CART Decision Tree

## Run directly in Jupyter

- Keep the extracted `datasets` folder beside the `notebooks` folder.
- Run every cell from top to bottom with **Shift + Enter**, or choose **Run → Run All Cells**.
- No manual input or internet download is required.
- If imports are missing, run `%pip install numpy pandas matplotlib seaborn scikit-learn` in a code cell, then rerun the failed cell.

**Dataset:** `datasets/sample_titanic.csv` is an included synthetic teaching dataset, not the official Titanic passenger dataset. Scores are illustrative.

In [ ]:
# Load the included sample dataset. No internet or manual input is needed.
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

possible_paths = [
    Path("sample_titanic.csv"),
    Path("../datasets/sample_titanic.csv"),
    Path("datasets/sample_titanic.csv"),
]
csv_file = next((p for p in possible_paths if p.exists()), None)
if csv_file is None:
    raise FileNotFoundError(
        "sample_titanic.csv not found. Keep the datasets folder beside notebooks."
    )

df = pd.read_csv(csv_file)
df = df.rename(columns={
    "PassengerId": "passenger_id", "Survived": "survived", "Pclass": "pclass",
    "Name": "name", "Sex": "sex", "Age": "age", "SibSp": "sibsp",
    "Parch": "parch", "Fare": "fare", "Embarked": "embarked", "Cabin": "cabin"
})
print("Loaded sample dataset:", csv_file)
print("Dataset shape:", df.shape)
display(df.head())


## Aim
Use a CART classification tree to predict Titanic survival. Gini impurity is used for splits.

## 1. Prepare data

In [ ]:
data = df[["survived","pclass","sex","age","sibsp","parch","fare","embarked"]].copy()
data["age"] = data["age"].fillna(data["age"].median())
data["fare"] = data["fare"].fillna(data["fare"].median())
data["embarked"] = data["embarked"].fillna(data["embarked"].mode()[0])
data = pd.get_dummies(data, columns=["sex","embarked"], drop_first=True, dtype=int)
X, y = data.drop(columns="survived"), data["survived"]
print("Features:", list(X.columns), "| X shape:", X.shape, "| y shape:", y.shape)


## 2. Train and evaluate

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
model = DecisionTreeClassifier(criterion="gini", max_depth=4, random_state=42)
model.fit(X_train,y_train)
y_pred = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test,y_pred),3))
print("\nConfusion Matrix:\n", confusion_matrix(y_test,y_pred))
print("\nClassification Report:\n", classification_report(y_test,y_pred,target_names=["Not Survived","Survived"]))
print("\nTree rules:\n", export_text(model,feature_names=list(X.columns)))


## 3. Draw the tree

In [ ]:
plt.figure(figsize=(18,9))
plot_tree(model,feature_names=list(X.columns),class_names=["Not Survived","Survived"],filled=True,rounded=True,fontsize=8)
plt.title("CART Decision Tree (Gini)"); plt.show()


## Expected output
Accuracy, confusion matrix, classification report, printed tree rules and tree diagram.